# Lab 0 — Observability & Evaluation Spine

**Curriculum §2 · Prerequisite for every other lab**

You build the measuring instrument *before* the thing being measured. By the end of this
notebook `evaluate(config, pipeline)` exists, and every later lab simply calls it.

### Why this ordering is non-negotiable
Without a harness you defend design choices with intuition. With one you say
*"+14 points recall@5"*. That difference is the whole curriculum.

### Before you start
```bash
docker compose up -d          # Langfuse, Qdrant, Redis, Phoenix
cp .env.example .env          # then paste your Langfuse + Groq keys
```


## ▶ Colab setup — run this cell first

1. Add your keys in Colab: **🔑 (left sidebar) → Secrets** → add `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, `GROQ_API_KEY` (toggle *Notebook access* on).
2. Free signups: **Langfuse** → cloud.langfuse.com · **Groq** → console.groq.com
3. Run the cell. It installs deps and pulls the shared `common/` modules.

> **If you see a `PIL._typing._Ink` import error:** run the cell, then **Runtime → Restart session**, then re-run. It's a Colab package clash, fixed by the Pillow upgrade above + a restart.


In [ ]:
# --- Colab bootstrap (safe to re-run) ---
import os, sys, subprocess, pathlib
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    subprocess.run('pip install -q -U Pillow'.split())  # fix Colab PIL._typing._Ink mismatch
    subprocess.run('pip install -q langfuse ragas sentence-transformers faiss-cpu '
                   'rank_bm25 langchain langchain-community langchain-groq '
                   'langchain-text-splitters langgraph pypdf pdfplumber PyMuPDF reportlab pandas'.split())

    # Pull the shared common/ package (config, corpus, golden, obs, scorers, harness).
    REPO = pathlib.Path('/content/genai_practical')
    if not (REPO/'common'/'harness.py').exists():
        # Option A: clone if you've pushed the repo to GitHub — set REPO_URL and uncomment:
        # subprocess.run(['git','clone','$REPO_URL', str(REPO)])
        # Option B: mount Google Drive where you unzipped the repo:
        try:
            from google.colab import drive; drive.mount('/content/drive')
            src = pathlib.Path('/content/drive/MyDrive/genai_practical')
            if src.exists(): REPO = src
        except Exception as e: print('Drive mount skipped:', e)
    sys.path.insert(0, str(REPO))

    # Keys from Colab Secrets -> env vars that common/config.py reads.
    try:
        from google.colab import userdata
        for k in ['LANGFUSE_PUBLIC_KEY','LANGFUSE_SECRET_KEY','GROQ_API_KEY']:
            v = userdata.get(k)
            if v: os.environ[k] = v
        os.environ.setdefault('LANGFUSE_HOST','https://cloud.langfuse.com')
    except Exception as e: print('Secrets not set — running offline. (', e, ')')
else:
    sys.path.insert(0, str(pathlib.Path.cwd().parent))   # local fallback

print('Environment:', 'Colab' if IN_COLAB else 'Local')


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # make `common` importable
from common.config import *
from common.corpus import DOCS, current_docs, SUPERSEDED_IDS
from common.golden import GOLDEN
from common.obs import observe, span_meta, trace_meta, make_config, flush, ENABLED
from common.harness import evaluate, leaderboard
print('Langfuse tracing:', 'ON' if ENABLED else 'OFF (labs still run)')
print('AS_OF:', AS_OF, '| docs in force:', [d['id'] for d in current_docs()])


## 1 · The corpus and its deliberate traps

`common/corpus.py` is engineered to break naive systems:

| Trap | Where | What it catches |
|---|---|---|
| **Version** | `pol-v2` vs `pol-v3` | A system that ignores `effective_date` quotes superseded policy |
| **Table** | `rate-1` | Flattened tables make rate questions unanswerable |
| **Jurisdiction** | `aml-1` (UK) vs `aml-2` (IE) | Wrong-country answers |
| **Refusal** | `q8` crypto question | Must refuse, not invent |


In [ ]:
import pandas as pd
pd.DataFrame(DOCS)[['id','doc','version','effective','jurisdiction','doctype']]


In [ ]:
# The trap made explicit: pol-v2 is still in the corpus but must NEVER be cited.
print('All docs      :', [d['id'] for d in DOCS])
print('In force today:', [d['id'] for d in current_docs()])
print('Superseded    :', SUPERSEDED_IDS)


## 2 · The golden set
8 questions to start. Expand toward ~100 for the real track — but keep the traps,
they carry most of the diagnostic signal.


In [ ]:
pd.DataFrame(GOLDEN)[['qid','q','source','version','difficulty','trap']]


## 3 · A minimal instrumented pipeline

Every stage is decorated with `@observe`, so Langfuse receives a nested span tree.
This baseline is deliberately simple — Labs R1–R8 replace one stage at a time.


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

_cache = {}
def embedder(name):
    if name not in _cache: _cache[name] = SentenceTransformer(name)
    return _cache[name]

@observe(name='retrieve')
def retrieve(query, cfg):
    pool = current_docs() if cfg['version_filter'] else DOCS
    m = embedder(cfg['embed_model'])
    M = m.encode([d['text'] for d in pool], normalize_embeddings=True)
    q = m.encode([query], normalize_embeddings=True)[0]
    sims = M @ q
    idx = np.argsort(-sims)[:cfg['k']]
    hits = [dict(pool[i], score=float(sims[i])) for i in idx]
    span_meta(returned=[h['id'] for h in hits],
              scores=[round(h['score'],3) for h in hits],
              version_filter=cfg['version_filter'], k=cfg['k'])
    return hits


In [ ]:
from langchain_groq import ChatGroq

llm   = ChatGroq(model=GEN_MODEL,   temperature=0)
judge = ChatGroq(model=JUDGE_MODEL, temperature=0)   # judge != generator in real labs

PROMPT = ("You are Meridian Bank's credit policy copilot. Answer ONLY from the context. "
          'Cite the chunk id in square brackets, e.g. [pol-v3]. '
          'If the context lacks the answer, reply exactly: INSUFFICIENT_CONTEXT.'
          '\n\nContext:\n{ctx}\n\nQuestion: {q}')

@observe(name='generate')
def generate(query, hits):
    ctx = '\n'.join(f"[{h['id']}] {h['text']}" for h in hits)
    out = llm.invoke(PROMPT.format(ctx=ctx, q=query)).content
    span_meta(n_context=len(hits))
    return out

@observe(name='rag.request')
def pipeline(query, cfg):
    hits = retrieve(query, cfg)
    ans  = generate(query, hits)
    trace_meta(tags=[cfg['hash']], config=cfg)
    return dict(answer=ans, hits=hits)


## 4 · Run three configurations

Run 3 deliberately **disables the version filter**. If the harness works,
`version_correct` collapses while every other metric stays healthy.


In [ ]:
cfgs = [
  make_config(embed_model=EMBED_MODELS['small'], k=2, version_filter=True),
  make_config(embed_model=EMBED_MODELS['base'],  k=2, version_filter=True),
  make_config(embed_model=EMBED_MODELS['base'],  k=2, version_filter=False),  # the trap
]
rows = [evaluate(c, pipeline, judge=judge) for c in cfgs]
leaderboard(rows)


## 5 · Read the result

| Signal | What it means |
|---|---|
| `version_correct` ~0 on run 3 | The harness caught a **compliance** failure |
| `faithfulness` still high on run 3 | The model faithfully quoted a *superseded* policy |
| `hit_at_k` barely moves | Generic RAG metrics would have called run 3 a success |

**That gap — between _faithful_ and _correct_ — is why domain metrics exist.**
A public RAG benchmark would have passed the configuration that gets your bank fined.

### Now open Langfuse → http://localhost:3000
1. **Traces** → open one → confirm the `rag.request → retrieve → generate` span tree
2. Check `retrieve` metadata shows returned ids + scores
3. Filter by the `cfg-…` tag to compare runs
4. Build the six dashboard panels from curriculum §2.3

---
### Exit gate
Change one parameter, re-run one cell, read the delta off a dashboard in under two minutes.

**Next →** `01_rag/lab01_ingestion_parsing.ipynb`
